소프트맥스 (다중 분류 로지스틱 회귀 모델)
M개의 입력을 받아 N개의 클래스로 출력하는 로지스틱 회귀 모델을 케라스로 구현해보도록 하겠습니다.
보통 다중 분류 로지스틱 회귀 모델을 소프트맥스(softmax)라고 부릅니다.
케라스에서 제공하는 MNIST 데이터를 사용하여 숫자를 0에서부터 9까지 분류해보도록 하겠습니다.

In [1]:
from keras.models import Sequential
from keras.layers import Dense, Activation
from keras.utils import to_categorical
##from keras.datasets import mnist

Using TensorFlow backend.


MNIST 손글씨 데이터를 다운로드 받아서 변수에 저장합니다.

In [2]:
##(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [3]:
import numpy as np

In [4]:
x1 = np.loadtxt('./X_train.txt')
x2 = np.loadtxt('./X_test.txt')
y_train = np.loadtxt('./y_train.txt')
y_test = np.loadtxt('./y_test.txt')
## user_id, pc_throttle, pc_brake, pc_steering, pc_speed, pc_lap_distance
X_train = x1.reshape(38372, 5)
X_test = x2.reshape(2955, 5)
print(str(X_train.shape))
print(str(X_test.shape))

(38372, 5)
(2955, 5)


손글씨 데이터(X_train, X_test)가 가로 28픽셀, 세로 28픽셀로 구성된 것을 확인할 수 있습니다.
학습에 사용될 X_train은 총 60000개의 데이터, 테스트에 사용될 X_test는 총 10000개의 데이터가 있습니다.

In [5]:
print("train data (count, row, column) : " + str(X_train.shape) )
print("test data  (count, row, column) : " + str(X_test.shape) )

train data (count, row, column) : (38372, 5)
test data  (count, row, column) : (2955, 5)


학습 데이터의 하나를 샘플로 보도록 하겠습니다. 아래 보시는 것처럼, 각각의 픽셀은 0부터 255까지의 값을 가지고 있습니다.

In [6]:
print(X_train[20000])

[0.    0.    0.004 0.503 0.102]


모델 학습 시작에 앞서, 데이터를 정규화합니다.
정규화는 입력값을 0부터 1의 값으로 변경하게 됩니다.
정규화된 입력값은 경사하강법으로 모델 학습 시, 보다 쉽고 빠르게 최적의 W,B를 찾는 데 도움을 줍니다.

In [7]:
X_train = X_train.astype('float32') 
X_test = X_test.astype('float32') 
##X_train /= 255 
##X_test /= 255

## 속도 81로 나눔
## 위치 5200으로 나눔
## 그래서 255 나누기 필요없음

아래의 명령어를 통해 정규화된 데이터를 확인할 수 있습니다.

In [8]:
print(X_train[0])

[0. 0. 0. 0. 0.]


y_train, y_test는 손글씨 데이터 (28*28 픽셀 데이터)에 해당하는 숫자를 나타냅니다.
y_train은 총 6만개, y_test는 총 1만개의 숫자를 가지고 있습니다.

In [9]:
print("train target (count) : " + str(y_train.shape) )
print("test target  (count) : " + str(y_test.shape) )

train target (count) : (38372,)
test target  (count) : (2955,)


In [10]:
print(y_test)
print(y_train)

[10. 10. 10. ... 10. 10. 10.]
[ 1.  1.  1. ... 10. 10. 10.]


아래의 코드를 실행하여, y_train과 y_test에서 샘플로 숫자를 출력해봅니다.

In [11]:
print("sample from train : " + str(y_train[0]) )
print("sample from test : " + str(y_test[0]) )

sample from train : 1.0
sample from test : 10.0


이번 실습에서는 28*28 픽셀의 지역적인 정보를 사용하지 않고, 단순히 정규화된 입력값만을 가지고,
숫자 분류를 할 것이기 때문에, 행과 열의 구분 없이, 단순히 784 길이의 배열로 데이터를 단순화시킵니다.

In [12]:
#input_dim = 784 #28*28 
#X_train = X_train.reshape(60000, input_dim) 
#X_test = X_test.reshape(10000, input_dim)

input_dim = 5 
X_train = X_train.reshape(38372, input_dim) 
X_test = X_test.reshape(2955, input_dim)

아래의 명령어를 실행하여, 현재 우리의 데이터가 2차원 데이터가 아닌 단순한 1차원 데이터로 변경된 것을 확인할 수 있습니다.

In [13]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(38372, 5)
(38372,)
(2955, 5)
(2955,)


학습 시, y값과의 cross entropy를 측정해야하므로, 아래의 코드를 실행하여 y를 one hot encoding으로 변환시켜줍니다.

In [14]:
num_classes = 11
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

아래 코드를 실행하여, 5였던 값이, one hot encoding으로 변환되어,
클래스 갯수만큼의 길이를 갖는 벡터로 변경이 되었고, 5에 해당되는 인덱스의 값만 1인 것을 확인할 수 있습니다.

In [15]:
print(y_train[0])
print(y_test[0])

[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


케라스의 Sequential()을 사용하여 간단하게 소프트맥스를 구현할 수 있습니다.
총 784개(28*28)의 입력을 받아서, 10개의 시그모이드 값을 출력하는 모델을 아래의 코드를 실행하여 구현합니다.

In [16]:
model = Sequential() 
model.add(Dense(input_dim=input_dim, units = 11, activation='softmax'))

모델의 학습을 진행합니다.
10개의 클래스로 분류할 것이기 때문에, categorical_crossentropy를 비용함수로 사용한 경사하강법으로 최적의 W와 biases를 학습합니다.

In [17]:
model.compile(optimizer='sgd', loss='categorical_crossentropy', metrics=['accuracy']) 
model.fit(X_train, y_train, batch_size=2048, epochs=100, verbose=0)

테스트를 진행하여, 정확도를 측정합니다.

In [18]:
score = model.evaluate(X_test, y_test) 
print('Test accuracy:', score[1])

2955/2955 [==============================] - 0s 27us/step
Test accuracy: 0.0


아래의 코드를 실행하여, 소프트맥스 모델의 구조를 쉽게 시각화 할 수 있습니다.
총 10개의 로지스틱회귀가 있고, 각 로지스틱회귀는 784개의 weight와 1개의 bias를 갖고 있기 때문에,
총 7850 (785*10)개의 Param이 있는 것을 보실 수 있습니다.

In [19]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_1 (Dense)              (None, 11)                66        
Total params: 66
Trainable params: 66
Non-trainable params: 0
_________________________________________________________________


첫번째 레이어에 존재하는 w1, w2,...,w784, b1, b2,..., b10은 아래의 명령어로 확인하실 수 있습니다.

In [20]:
model.layers[0].weights

[<tf.Variable 'dense_1/kernel:0' shape=(5, 11) dtype=float32, numpy=
 array([[-0.75875807,  0.3856374 , -0.11077613,  0.30319533,  0.43441287,
          0.14466314,  0.49940646,  0.32237363,  0.10312622, -0.15392321,
         -0.1256147 ],
        [-0.6545198 , -0.05852519,  0.16664696, -0.05742028, -0.26285166,
         -0.07276115, -0.02311441, -0.32285938, -0.4000412 ,  0.22223239,
         -0.01774512],
        [-0.24968937,  0.48460284,  0.7132876 ,  0.4591732 ,  0.03272627,
         -0.5160493 ,  0.05827053,  0.39620388,  0.22362886, -0.06347191,
         -0.24294288],
        [-0.17023215,  0.23609255,  0.30901772, -0.38897568, -0.50916827,
         -0.05149037,  0.6731849 , -0.01223505, -0.38014647,  0.18552636,
         -0.06534759],
        [-0.5581009 , -0.34247681,  0.7982956 , -0.00789466, -0.21614163,
          0.52388126, -0.38523066,  0.3202781 ,  0.3025621 ,  0.4432083 ,
          0.2316948 ]], dtype=float32)>,
 <tf.Variable 'dense_1/bias:0' shape=(11,) dtype=float32, 